# Phase 1: Qwen3.5-35B-A3B Thinking Traces — SAE Training Data

**Goal**: generate thinking rollouts + cache per-sentence L17 activations for SAE training. Exact replication of Venhoff et al. 2025 (arXiv:2510.07364) recipe.

**Output**: HF dataset `caiovicentino1/qwen35-a3b-thinking-traces` with ~70k sentences × L17 activations.

**Recipe**:
1. Load `Qwen/Qwen3.5-35B-A3B` (thinking model, NOT -Base)
2. Sample 2000 prompts from MMLU-Pro
3. Generate thinking rollouts (max 512 new tokens, temp 0.6)
4. Capture L17 residual activations during generation
5. Split thinking text into sentences, mean-pool activations per sentence
6. Save sentences + activations + metadata to HF dataset

**Budget**: ~8-10h on RTX 6000 Blackwell 96GB.  
**Target**: 2000 prompts × ~35 sentences = ~70k sentences (paper used 430k, but small SAE with n=15 needs far less).

## Cell 1 — Install

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers, datasets
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping: {e})')

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/qwen35_thinking'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load Qwen3.5-35B-A3B (THINKING model)

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import HfApi, create_repo, snapshot_download

MODEL_ID = 'Qwen/Qwen3.5-35B-A3B'  # thinking model
HF_DATASET_ID = 'caiovicentino1/qwen35-a3b-thinking-traces'
SAE_LAYER = 17  # ~42% depth for 40-layer model (matches paper's Qwen2.5-32B layer 27/64)

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model: {MODEL_ID}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Create HF dataset repo
api = HfApi()
try:
    create_repo(HF_DATASET_ID, repo_type='dataset', private=False, exist_ok=True)
    print(f'✅ HF dataset ready: https://huggingface.co/datasets/{HF_DATASET_ID}')
except Exception as e:
    print(f'repo creation: {e}')

## Cell 3 — Load MMLU-Pro prompts

In [ ]:
from datasets import load_dataset

# MMLU-Pro test set has ~12k items
mmlu = load_dataset('TIGER-Lab/MMLU-Pro', split='test')
print(f'MMLU-Pro test size: {len(mmlu)}')

# Sample 2000 for initial run
N_PROMPTS = 2000
random.seed(42)
sample_idx = random.sample(range(len(mmlu)), min(N_PROMPTS, len(mmlu)))
prompts = [mmlu[i] for i in sample_idx]
print(f'Sampled {len(prompts)} prompts')

# Format prompt — plain CoT style per paper
def format_prompt(ex):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(ex['options']))
    return (
        f"Task: Answer the question below. Explain your reasoning step by step.\n\n"
        f"Question: {ex['question']}\n\nOptions:\n{choices}\n\n"
        f"Step by step answer:")

# Preview
print(f'\nSample prompt:')
print(format_prompt(prompts[0])[:500])

## Cell 4 — Setup L17 activation capture + sentence splitting

In [ ]:
# Hook to capture L17 residual activations during forward
captured_acts = {'value': None}
def capture_l17(module, inp, out):
    hidden = out[0] if isinstance(out, tuple) else out
    captured_acts['value'] = hidden.detach().clone()  # [B, T, d_model]

h_capture = get_layer_module(model, SAE_LAYER).register_forward_hook(capture_l17)
print(f'✅ L{SAE_LAYER} activation capture hook installed')

# Sentence splitter — simple regex matching paper's approach
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text, min_len=10):
    """Split text into sentences. Filter out very short fragments."""
    sents = [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
    return [s for s in sents if len(s) >= min_len]

def sentence_token_ranges(full_text_tokens, tokenizer, sentences):
    """For each sentence, find its token range in the full decoded text.
    Returns list of (start_idx, end_idx) tuples or None if not found."""
    # Decode each prefix incrementally to find token boundaries
    # Simpler approach: decode full + use char offset from tokenizer
    ranges = []
    cursor = 0
    for sent in sentences:
        # Find sentence in full tokens via substring search on decoded
        pos = tokenizer.decode(full_text_tokens).find(sent, cursor)
        if pos < 0:
            ranges.append(None); continue
        cursor = pos + len(sent)
        # Convert char positions to token indices via tokenizer.encode of prefix
        prefix = tokenizer.decode(full_text_tokens)[:pos]
        suffix_stop = tokenizer.decode(full_text_tokens)[:pos + len(sent)]
        # Approximate: use tokenizer to encode
        start_tok = len(tokenizer.encode(prefix, add_special_tokens=False))
        end_tok = len(tokenizer.encode(suffix_stop, add_special_tokens=False))
        ranges.append((start_tok, end_tok))
    return ranges

print('✅ helpers ready')

## Cell 5 — Generate thinking rollouts + cache sentence activations

~2000 prompts × ~10-15s generation + capture each = **~8-10h on RTX 6000**.
Incremental save every 100 prompts (crash-resistant).

In [ ]:
MAX_NEW = 512
TEMP = 0.6
SAVE_EVERY = 100

all_sentences = []  # list of dicts: {prompt_idx, sentence, activation (list), category_id}
t0 = time.time()

for pi, ex in enumerate(tqdm(prompts, desc='thinking rollouts')):
    try:
        p = format_prompt(ex)
        # Use chat template to enable thinking mode
        chat_p = tok.apply_chat_template([{'role':'user','content':p}],
                                          tokenize=False, add_generation_prompt=True)
        ids = tok(chat_p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 3500: continue
        prompt_len = ids.shape[1]

        # Generate with sampling (thinking mode)
        torch.manual_seed(42 + pi)
        with torch.no_grad():
            gen = model.generate(
                input_ids=ids, attention_mask=torch.ones_like(ids),
                max_new_tokens=MAX_NEW, do_sample=True, temperature=TEMP,
                pad_token_id=tok.pad_token_id, use_cache=True)
        full_ids = gen[0]
        response_ids = full_ids[prompt_len:].tolist()
        response_text = tok.decode(response_ids, skip_special_tokens=True)

        # Re-forward full sequence to capture activations
        full_seq = full_ids.unsqueeze(0)
        with torch.no_grad():
            _ = model(input_ids=full_seq, attention_mask=torch.ones_like(full_seq))
        l17_acts = captured_acts['value']  # [1, T, d_model]
        response_acts = l17_acts[0, prompt_len:, :]  # [T_resp, d_model]

        # Split response into sentences
        sentences = split_sentences(response_text)
        if len(sentences) == 0: continue

        # Find token ranges for each sentence
        ranges = sentence_token_ranges(response_ids, tok, sentences)

        # For each valid sentence, mean-pool activations over its token range
        for sent, rng in zip(sentences, ranges):
            if rng is None: continue
            s, e = rng
            if s >= response_acts.shape[0] or e > response_acts.shape[0] or s >= e:
                continue
            sent_act = response_acts[s:e, :].mean(dim=0).float().cpu().numpy()
            all_sentences.append(dict(
                prompt_idx=pi,
                sentence=sent,
                activation=sent_act.tolist(),
                token_range=(s, e)))

        if (pi + 1) % SAVE_EVERY == 0:
            elapsed = (time.time() - t0) / 60
            print(f'  {pi+1}/{len(prompts)} | {len(all_sentences)} sentences | {elapsed:.1f} min | ETA {elapsed/(pi+1)*(len(prompts)-pi-1):.0f} min')
            # Save incremental
            with open(OUT / 'sentences_partial.pkl', 'wb') as f:
                pickle.dump(all_sentences, f)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        print(f'prompt {pi}: {e}'); continue

h_capture.remove()

# Final save
with open(OUT / 'sentences.pkl', 'wb') as f:
    pickle.dump(all_sentences, f)
print(f'\n✅ Done in {(time.time()-t0)/60:.1f} min')
print(f'Total sentences: {len(all_sentences)}')
print(f'Avg per prompt: {len(all_sentences)/len(prompts):.1f}')

## Cell 6 — Upload to HF dataset

In [ ]:
# Convert to compact numpy format + save as safetensors for efficient storage
import numpy as np
from safetensors.numpy import save_file

d_model = len(all_sentences[0]['activation'])
print(f'd_model: {d_model}')

activations = np.stack([np.array(s['activation'], dtype=np.float16) for s in all_sentences])  # [N, d]
sentence_texts = [s['sentence'] for s in all_sentences]
prompt_idxs = np.array([s['prompt_idx'] for s in all_sentences], dtype=np.int32)

print(f'activations shape: {activations.shape}, dtype {activations.dtype}')
print(f'Size: {activations.nbytes / 1e6:.1f} MB')

# Save
save_file({'activations': activations, 'prompt_idxs': prompt_idxs},
          str(OUT / 'activations.safetensors'))
with open(OUT / 'sentences.json', 'w') as f:
    json.dump(sentence_texts, f)

# Upload to HF
try:
    api.upload_file(path_or_fileobj=str(OUT / 'activations.safetensors'),
                    path_in_repo='activations.safetensors',
                    repo_id=HF_DATASET_ID, repo_type='dataset',
                    commit_message=f'Phase 1: {len(all_sentences)} sentences from {len(prompts)} MMLU-Pro prompts')
    api.upload_file(path_or_fileobj=str(OUT / 'sentences.json'),
                    path_in_repo='sentences.json',
                    repo_id=HF_DATASET_ID, repo_type='dataset')
    # README
    readme = f'''---
license: mit
tags:
  - mechanistic-interpretability
  - sparse-autoencoders
  - qwen3.5
  - thinking-models
size_categories:
  - 10K<n<100K
---

# Qwen3.5-35B-A3B Thinking Traces — SAE Training Data

Per-sentence L{SAE_LAYER} residual activations from Qwen/Qwen3.5-35B-A3B generating CoT on MMLU-Pro.

## Stats

- Model: `Qwen/Qwen3.5-35B-A3B`
- Layer: L{SAE_LAYER} residual (~42% depth of 40-layer hybrid MoE)
- Prompts: {len(prompts)} from MMLU-Pro test
- Sentences: {len(all_sentences)}
- d_model: {d_model}
- Activation dtype: float16

## Purpose

Replication of Venhoff et al. 2025 (arXiv:2510.07364) "Base Models Know How to Reason, Thinking Models Learn When" applied to hybrid MoE+GDN+Gated-Attn architecture.

Phase 1 of 3 (data generation). Next: tiny TopK SAE training (n=15, k=3) to cluster reasoning categories.

## Load

```python
from safetensors.numpy import load_file
import json
from huggingface_hub import snapshot_download

path = snapshot_download('{HF_DATASET_ID}', repo_type='dataset')
data = load_file(f'{{path}}/activations.safetensors')
sentences = json.load(open(f'{{path}}/sentences.json'))
```
'''
    with open(OUT / 'README.md', 'w') as f:
        f.write(readme)
    api.upload_file(path_or_fileobj=str(OUT / 'README.md'),
                    path_in_repo='README.md',
                    repo_id=HF_DATASET_ID, repo_type='dataset')
    print(f'\n✅ Uploaded to https://huggingface.co/datasets/{HF_DATASET_ID}')
except Exception as e:
    print(f'upload failed: {e}')